<a href="https://colab.research.google.com/github/netsetos/agentic-ai-weekend-gcp-learners/blob/main/module-07-mcp-and-cloud-run/lesson-7.2-cloud-run-deploy/notebooks/GCP_Capstone_7.2_CloudRunDeploy.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 7.2 Deploy the Agent Surface — Cloud Run, IAM, the Roster, Scale to Zero, the Toolbox
**Netsetos GenAI Engineering — GCP Capstone** · Module 7 · rebuilt on the live lane, 8 September 2026

7.1's server becomes `documind-mcp`: a fourth Cloud Run service of the kit, as its own account, on the tenant rosters. Then three calls that prove the boundary is identity and not the network, a cold-start measurement, the kit's smoke leg, and the second server - the MCP Toolbox over Module 5's warehouse - with the two defects of its first version fixed.


## Setup


In [ ]:
!pip install -q fastmcp==3.4.7 google-auth==2.57.1 google-cloud-firestore==2.30.0 requests==2.34.2

from google.colab import auth
auth.authenticate_user()

PROJECT_ID = "documind-ai-YOUR-ID"   # CHANGE THIS: the project the lane runs in
REGION     = "us-central1"
KIT        = "/content/agentic-ai-weekend-gcp"
BRANCH     = "feat/lesson-4.8-live-evals"

import json, os, subprocess, sys, time
import google.auth
from google.auth import impersonated_credentials
from google.auth.transport.requests import AuthorizedSession, Request

if not os.path.isdir(KIT):
    subprocess.run(["git", "clone", "--depth", "1", "-q", "-b", BRANCH,
                    "https://github.com/netsetos/agentic-ai-weekend-gcp", KIT], check=True)
sys.path.insert(0, f"{KIT}/deploy")
subprocess.run(["gcloud", "config", "set", "project", PROJECT_ID], capture_output=True)

creds, _ = google.auth.default()
NUMBER  = AuthorizedSession(creds).get(f"https://cloudresourcemanager.googleapis.com/v1/projects/{PROJECT_ID}").json()["projectNumber"]
GIT_SHA = subprocess.run(["git", "-C", KIT, "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
MCP_URL = f"https://documind-mcp-{NUMBER}.{REGION}.run.app"      # deterministic, like every service's (eventarc.tf)
API_URL = f"https://documind-api-{NUMBER}.{REGION}.run.app"
MCP_SA, MEMBER_SA, OUTSIDER_SA = (f"documind-{n}-sa@{PROJECT_ID}.iam.gserviceaccount.com" for n in ("mcp", "ui", "outsider"))

# The deploy block below reads exactly what the kit's Makefile hands its deploy scripts.
os.environ.update({"PROJECT": PROJECT_ID, "GIT_SHA": GIT_SHA, "REGION": REGION,
                   "PROJECT_NUMBER": NUMBER, "MIN_INSTANCES": "0"})
print("image tag:", GIT_SHA, "| service URL will be:", MCP_URL)


## Cell 1: The account and its five roles
Not `documind-outsider-sa` - that account is the eval gate's outsider on purpose. Not `documind-ui-sa` - two services under one identity blur 12.3's audit log. A fourth account, declared where the other four are, with the roles it needs and none it does not.


In [ ]:
# The account is terraform's (deploy/terraform/sa.tf, owned by 12.1's notebook): one service
# account, five project roles, actAs for the CI/CD identity. Read the block the kit carries:
sa_tf = open(f"{KIT}/deploy/terraform/sa.tf", encoding="utf-8").read()
start = sa_tf.index("# The MCP server (7.1-7.2)")
print(sa_tf[start:start + 720])
print("  mcp_roles:", [l.strip() for l in sa_tf[sa_tf.index("mcp_roles = ["):].split("]")[0].splitlines() if l.strip().startswith('"')])

# Does it exist yet? On a terraform-managed project the honest way to create it is `make up`
# (or `make plan` + apply) in Cloud Shell: additive, one account and its bindings. Creating it
# with gcloud on such a project makes drift that `make drift` will report.
have = subprocess.run(["gcloud", "iam", "service-accounts", "describe", MCP_SA, "--project", PROJECT_ID],
                      capture_output=True, text=True).returncode == 0
print("\ndocumind-mcp-sa exists:", have)
if not have:
    print("-> Cloud Shell:  cd deploy && make up PROJECT=" + PROJECT_ID + "   (terraform creates it; nothing else changes)")


In [ ]:
# ONLY for a project you do not manage with terraform (a scratch project): the gcloud equivalent
# of sa.tf's block. On the lane, leave this False and use `make up`.
CREATE_WITH_GCLOUD = False
if CREATE_WITH_GCLOUD:
    subprocess.run(["gcloud", "iam", "service-accounts", "create", "documind-mcp-sa", "--project", PROJECT_ID,
                    "--display-name", "DocuMind MCP server (agent surface on Cloud Run)"], check=False)
    for role in ("roles/run.invoker", "roles/datastore.viewer", "roles/logging.logWriter",
                 "roles/cloudtrace.agent", "roles/storage.objectCreator"):
        subprocess.run(["gcloud", "projects", "add-iam-policy-binding", PROJECT_ID, "--quiet",
                        "--member", f"serviceAccount:{MCP_SA}", "--role", role], capture_output=True, check=True)
    print("created with gcloud - remember this project is now off terraform for this account")


## Cell 2: Build the image
The kit's way: Cloud Build from the `deploy/` context with the Dockerfile named, so `shared/` lands beside the server. This is `make build SERVICES_lean=mcp` without make.


In [ ]:
# Build the image the way the kit builds every image: Cloud Build, the deploy/ context, the
# Dockerfile named. This is `make build SERVICES_lean=mcp` without make.
IMAGE = f"{REGION}-docker.pkg.dev/{PROJECT_ID}/documind/mcp:{GIT_SHA}"
r = subprocess.run(["gcloud", "builds", "submit", ".", "--project", PROJECT_ID, "--quiet", "--config=cloudbuild.yaml",
                    f"--substitutions=_IMAGE={IMAGE},_DOCKERFILE=services/mcp/Dockerfile"],
                   cwd=f"{KIT}/deploy", capture_output=True, text=True)
print("build:", "ok" if r.returncode == 0 else r.stderr[-800:])
print("image:", IMAGE)


## Cell 3: The deploy block — the kit's, not a notebook's
One definition. The extractor turns this heredoc into `deploy/commands/lesson-7.2.sh`, which `make deploy-services` runs for the `mcp` service in both profiles. Read the environment variables: `SELF_URL` is the audience every caller's token must carry, `RAG_API_URL` is where the one `retrieve()` goes, and the two `run.invoker` grants say who may knock.


In [ ]:
# The deploy block. It is a heredoc for the same reason 7.1's server is: the extractor turns it into
# deploy/commands/lesson-7.2.sh, the file `make deploy-services` runs for the mcp service - one
# definition, used by Cloud Shell and by this notebook. Variables come from the environment
# (Setup), the way the Makefile hands them to every deploy script.
DEPLOY = r'''
# The MCP server (7.1): the lane's agent surface. Its own account (sa.tf), no unauthenticated calls,
# the API's URL and its own URL in the environment (SELF_URL is the audience every caller's token
# must carry), stateless streamable HTTP so nothing is pinned to an instance. No VPC connector: it
# calls the API on the public run.app hostname like the UI does, and the boundary is identity.
# RAG_TIMEOUT_S=90: the shared tool layer waits 20 s by default, and a cold API plus a Gemini answer is
# longer than that - the first live call timed out. 90 s is what the eval gate itself waits.
gcloud run deploy documind-mcp \
  --image=${REGION:-us-central1}-docker.pkg.dev/$PROJECT/documind/mcp:$GIT_SHA \
  --region=${REGION:-us-central1} --platform=managed \
  --no-allow-unauthenticated \
  --ingress=all \
  --memory=1Gi --cpu=1 --concurrency=40 --timeout=120 \
  --min-instances=${MIN_INSTANCES:-0} --max-instances=10 \
  --cpu-boost --execution-environment=gen2 \
  --service-account=documind-mcp-sa@$PROJECT.iam.gserviceaccount.com \
  --set-env-vars="^|^GOOGLE_CLOUD_PROJECT=$PROJECT|DOCUMIND_PROFILE=gcp|RAG_API_URL=https://documind-api-$PROJECT_NUMBER.${REGION:-us-central1}.run.app|SELF_URL=https://documind-mcp-$PROJECT_NUMBER.${REGION:-us-central1}.run.app|FASTMCP_STATELESS_HTTP=true|RAG_TIMEOUT_S=90"

# Who may KNOCK: IAM invoker on this service. The operators' accounts (7.3's notebook mints as the
# UI's), and the eval gate's outsider - deliberately, so the third call in 7.2 is refused by the
# ROSTER and not by the network. An agent service you add later gets the same line for its account.
for who in documind-ui-sa documind-outsider-sa; do
  gcloud run services add-iam-policy-binding documind-mcp \
    --region=${REGION:-us-central1} --project=$PROJECT \
    --member="serviceAccount:$who@$PROJECT.iam.gserviceaccount.com" --role=roles/run.invoker --quiet
done
'''

r = subprocess.run(["bash", "-ec", DEPLOY], env=os.environ, capture_output=True, text=True)
print(r.stdout[-600:], r.stderr[-1200:] if r.returncode else "")
print("deployed:", MCP_URL if r.returncode == 0 else "FAILED - read the tail above")


## Cell 4: The roster
The server calls the API *as* `documind-mcp-sa`, and the API checks that account on the tenant's roster like any other caller. So the account sits on the three golden tenants beside the UI's. `make roster` does this in Cloud Shell; this cell is the same write.


In [ ]:
# The roster. The server calls the API AS documind-mcp-sa, and the API checks that account on the
# tenant's roster like any other caller - so the account sits on the three golden tenants beside the
# UI's. `make roster` does this in Cloud Shell; this is the same call (shared/tenancy.py, the
# operator's write side) from here.
from shared import tenancy
for tenant in ("acme", "zeta", "globex"):
    if not tenancy.is_member(MCP_SA, tenant):
        tenancy.add_member(tenant, MCP_SA)
    print(f"{tenant:7} members: {sorted(m.split('@')[0] for m in tenancy.list_members(tenant))}")


## Cell 5: Three calls
No token: the platform refuses before the server runs. The outsider: IAM lets it knock, the roster refuses. A member naming the tenant: through the server, through the API, back with citations. The second call is the one to watch - it is F10's shape on a new surface, and it is a replay you can run in the room.


In [ ]:
from fastmcp import Client
from fastmcp.client.transports import StreamableHttpTransport
import requests

SCOPE = ["https://www.googleapis.com/auth/cloud-platform"]
def id_token_as(service_account: str, audience: str) -> str:
    source, _ = google.auth.default()
    target = impersonated_credentials.Credentials(source_credentials=source, target_principal=service_account, target_scopes=SCOPE)
    idc = impersonated_credentials.IDTokenCredentials(target, target_audience=audience, include_email=True)
    idc.refresh(Request())
    return idc.token

async def call(token, name, args=None):
    headers = {"Authorization": f"Bearer {token}"} if token else {}
    async with Client(StreamableHttpTransport(f"{MCP_URL}/mcp", headers=headers)) as c:
        r = await c.call_tool(name, args or {})
        return r.data if getattr(r, "data", None) is not None else r

Q = "After how many years of continuous service does gratuity become payable?"

# 1. no token: Cloud Run's IAM ingress answers, before any of this server's code runs
r = requests.get(f"{MCP_URL}/health", timeout=30)
print("1. no token         ->", r.status_code, "(the platform, not the server: nobody may knock without a token)")

# 2. the outsider: IAM lets it knock (we granted invoker), the roster refuses - authenticated is not authorised
try:
    await call(id_token_as(OUTSIDER_SA, MCP_URL), "retrieve", {"query": Q})
    print("2. outsider         -> UNEXPECTED answer")
except Exception as e:
    print("2. outsider         -> refused:", str(e)[:100])

# 3. a member, naming the tenant: through the server, through the API, back with citations
out = await call(id_token_as(MEMBER_SA, MCP_URL), "retrieve", {"query": Q, "tenant": "acme"})
print("3. member (acme)    ->", out.get("answerable"), out.get("confidence"), len(out.get("citations") or []), "citations")
print("   ", (out.get("answer") or "")[:160])


## Cell 6: Scale to zero, measured


In [ ]:
# Scale to zero and what a cold start costs. min-instances=0 means the first call after idle time
# pays for a container start plus the SDK imports; the second call does not. Two timings, no theory.
import time
tok = id_token_as(MEMBER_SA, MCP_URL)
for label in ("first (maybe cold)", "second (warm)"):
    t0 = time.perf_counter()
    async with Client(StreamableHttpTransport(f"{MCP_URL}/mcp", headers={"Authorization": f"Bearer {tok}"})) as c:
        n = len(await c.list_tools())
    print(f"{label:20} {(time.perf_counter() - t0) * 1000:7.0f} ms  ({n} tools)")
print("The UI keeps a floor of one instance because a person waits; an agent surface at zero costs nothing idle.")


## Cell 7: The smoke leg
`deploy/smoke/smoke_mcp.py` ships with the kit: health, the four tools, a cited answer as a member, the outsider refused. `make smoke-mcp` in Cloud Shell; the same file run from here.


In [ ]:
# The smoke leg the kit ships for this service - deploy/smoke/smoke_mcp.py, `make smoke-mcp` in
# Cloud Shell. Four checks: health, the four tools, a cited answer as a member, the outsider refused
# by the roster. Reference block for the extractor (commands/lesson-7.2.sh, SMOKE section):
SMOKE = r'''
# from deploy/, after `pip install --user fastmcp==3.4.7`
make smoke-mcp PROJECT=$PROJECT

# the same by hand: health through the IAM ingress, then tools/list as JSON-RPC over streamable HTTP
TOK=$(gcloud auth print-identity-token --include-email --impersonate-service-account=documind-ui-sa@$PROJECT.iam.gserviceaccount.com --audiences=https://documind-mcp-$PROJECT_NUMBER.us-central1.run.app)
curl -sSf -H "Authorization: Bearer $TOK" https://documind-mcp-$PROJECT_NUMBER.us-central1.run.app/health
curl -sS -H "Authorization: Bearer $TOK" -H "Content-Type: application/json" -H "Accept: application/json, text/event-stream" \
  -d '{"jsonrpc":"2.0","id":1,"method":"initialize","params":{"protocolVersion":"2025-06-18","capabilities":{},"clientInfo":{"name":"curl","version":"0"}}}' \
  https://documind-mcp-$PROJECT_NUMBER.us-central1.run.app/mcp
'''

env = {**os.environ, "DOCUMIND_MCP_URL": MCP_URL, "DOCUMIND_IMPERSONATE_SA": MEMBER_SA, "DOCUMIND_OUTSIDER_SA": OUTSIDER_SA}
r = subprocess.run([sys.executable, "smoke/smoke_mcp.py"], cwd=f"{KIT}/deploy", env=env, capture_output=True, text=True)
print(r.stdout[-1500:], r.stderr[-400:])


## What the first live deploy found
Two things, both the lane's and neither the notebook's - found by `make smoke-mcp`, fixed at their source, and kept here because the next surface will meet them too.

1. **A 422 from the API on the first warm call.** `brain` is an enumerated label in the API's request schema (`langchain | langgraph | adk | direct | ui`), so the usage row can compare harnesses. A new surface has to be added to that list - one line in 12.2's schemas and an API-only redeploy, the loop's own move - and the warehouse can now tell an MCP call from a person's. The list is closed on purpose: an unknown label is a typo, not a new row.
2. **A timeout on the first cold call.** The shared tool layer waits 20 seconds by default; a cold API plus a Gemini answer is longer than that. The service sets `RAG_TIMEOUT_S=90` in its environment - the number the eval gate itself waits - and the deploy block above carries it.

Read the server's own log for either: `retrieve failed: 422 Client Error` and `Read timed out (read timeout=20.0)` were the two lines, one command away.


### Replay them
Both findings replay in the room, the way 4.8's findings do - a traffic flip and an environment flip, nothing rebuilt:

```bash
# The 422, replayed: the API revision that was serving before the fix is still deployed with no traffic
# (Cloud Run keeps revisions; git keeps the tag demo/state-5-brain-label-closed at the same code).
PROJECT=documind-ai-YOUR-ID; REGION=us-central1
PREV=$(gcloud run revisions list --service=documind-api --region=$REGION --project=$PROJECT --format='value(name)' --sort-by=~metadata.creationTimestamp | sed -n 2p)
gcloud run services update-traffic documind-api --region=$REGION --project=$PROJECT --to-revisions=$PREV=100
make smoke-mcp PROJECT=$PROJECT 2>&1 | grep -E "PASS|FAIL"          # retrieve: document retrieval is unavailable
gcloud logging read 'resource.type="cloud_run_revision" AND resource.labels.service_name="documind-mcp" AND textPayload:"retrieve failed"' --project=$PROJECT --limit=1 --format='value(textPayload)'
gcloud run services update-traffic documind-api --region=$REGION --project=$PROJECT --to-latest
make smoke-mcp PROJECT=$PROJECT 2>&1 | grep -E "PASS|FAIL"          # 4 of 4 again

# The timeout, replayed: the shared tool layer's default wait, on a cold API (idle a few minutes first)
gcloud run services update documind-mcp --region=$REGION --project=$PROJECT --update-env-vars RAG_TIMEOUT_S=20 --quiet
make smoke-mcp PROJECT=$PROJECT 2>&1 | grep -E "retrieve"           # Read timed out (read timeout=20.0) - on a cold API
gcloud run services update documind-mcp --region=$REGION --project=$PROJECT --update-env-vars RAG_TIMEOUT_S=90 --quiet
```

The API revision that refused `mcp` is still deployed with no traffic, and git tags the same code as `demo/state-5-brain-label-closed`; the timeout needs the API cold, so let it idle a few minutes first.


## Cell 8: The second server — MCP Toolbox for Databases
Tools generated from a config, no code, over the warehouse Module 5 builds. Two defects of this lesson's first version are fixed in the config: the analytics query counted a column that does not exist (`source_file`; the kit's table has `source_uri`), and the source named a location that matched neither Module 5's dataset nor the kit's terraform. The Cloud SQL source is gone: nothing on the lean lane creates that instance.


In [ ]:
# THE SECOND SERVER: MCP Toolbox for Databases - tools generated from a config, no code. Where this
# server is the lane's OPERATIONS, the Toolbox is the lane's WAREHOUSE: analytics over the tables
# Module 5 builds. Two things differ from the first version of this lesson: the column is
# source_uri (the kit's chunk_metadata has no source_file - BigQuery refused the old query), and
# the dataset location is ONE variable shared with Module 5 (BQ_LOCATION), because a Toolbox
# source must match the dataset's location or every query fails. No Cloud SQL source: nothing
# on the lean lane creates that instance; the full profile's Postgres is documind-checkpoint (8.5).
BQ_LOCATION = "US"          # Module 5's choice (5.3 creates rag_data there); the kit's terraform uses asia-south1 - pick ONE per project
tools_yaml = f"""kind: source
name: documind-bq
type: bigquery
project: {PROJECT_ID}
location: {BQ_LOCATION}
---
kind: tool
name: query-document-analytics
type: bigquery-sql
source: documind-bq
description: |
  Aggregate document analytics from the warehouse Module 5 built: chunk, token and page
  counts by document type over a recent window. Use for questions about volumes or trends
  - NOT for retrieving the text of a document (that is DocuMind's retrieve tool).
parameters:
  - name: tenant_id
    type: string
    description: Which tenant's corpus to report on.
  - name: days
    type: integer
    description: How many days back to aggregate (1-90).
statement: |
  SELECT doc_type,
         COUNT(DISTINCT source_uri) AS documents,
         COUNT(*) AS chunks,
         SUM(token_count) AS tokens,
         ROUND(SUM(page_end - page_start + 1), 0) AS pages
  FROM `{PROJECT_ID}.rag_data.chunk_metadata`
  WHERE tenant_id = @tenant_id
    AND DATE(featured_at) >= DATE_SUB(CURRENT_DATE(), INTERVAL @days DAY)
  GROUP BY doc_type
  ORDER BY chunks DESC
---
kind: tool
name: predict-document-category
type: bigquery-sql
source: documind-bq
description: |
  Predict a document's type with the BQML classifier trained in Module 5.1
  (ml_models.doc_classifier). It scores STRUCTURAL features - page count, word count,
  chunk statistics, whether the document has tables or images - not the text itself.
parameters:
  - name: page_count
    type: integer
  - name: total_word_count
    type: integer
  - name: avg_chunk_size
    type: float
  - name: chunk_count
    type: integer
  - name: file_size_mb
    type: float
  - name: has_tables
    type: boolean
  - name: has_images
    type: boolean
statement: |
  SELECT predicted_doc_type, predicted_doc_type_probs
  FROM ML.PREDICT(MODEL `{PROJECT_ID}.ml_models.doc_classifier`,
    (SELECT @page_count AS page_count, @total_word_count AS total_word_count,
            @avg_chunk_size AS avg_chunk_size, @chunk_count AS chunk_count,
            @file_size_mb AS file_size_mb, @has_tables AS has_tables, @has_images AS has_images))
"""
open("tools.yaml", "w", encoding="utf-8").write(tools_yaml)
print("tools.yaml:", tools_yaml.count("kind: tool"), "tools,", tools_yaml.count("kind: source"), "source - needs Module 5's rag_data.chunk_metadata and ml_models.doc_classifier")


In [ ]:
# Deploy the Toolbox - the official image, the config as a secret, its own least-privilege account.
# Off by default: it needs Module 5's tables to be useful, and it is a second Cloud Run service.
DEPLOY_TOOLBOX = False
if DEPLOY_TOOLBOX:
    TB = f"""
gcloud secrets create tools-yaml --project=$PROJECT --data-file=tools.yaml 2>/dev/null || gcloud secrets versions add tools-yaml --project=$PROJECT --data-file=tools.yaml
gcloud iam service-accounts create documind-toolbox-sa --project=$PROJECT --display-name="DocuMind Toolbox (BigQuery tools)" 2>/dev/null || true
gcloud secrets add-iam-policy-binding tools-yaml --project=$PROJECT --member="serviceAccount:documind-toolbox-sa@$PROJECT.iam.gserviceaccount.com" --role=roles/secretmanager.secretAccessor --quiet
for role in roles/bigquery.jobUser roles/bigquery.dataViewer; do
  gcloud projects add-iam-policy-binding $PROJECT --member="serviceAccount:documind-toolbox-sa@$PROJECT.iam.gserviceaccount.com" --role=$role --quiet >/dev/null
done
gcloud run deploy documind-toolbox --project=$PROJECT --region=$REGION \
  --image=us-central1-docker.pkg.dev/database-toolbox/toolbox/toolbox:latest \
  --service-account=documind-toolbox-sa@$PROJECT.iam.gserviceaccount.com \
  --no-allow-unauthenticated --min-instances=0 \
  --set-secrets="/app/tools.yaml=tools-yaml:latest" \
  --args="--tools-file=/app/tools.yaml","--address=0.0.0.0","--port=8080"
gcloud run services add-iam-policy-binding documind-toolbox --project=$PROJECT --region=$REGION \
  --member="serviceAccount:documind-ui-sa@$PROJECT.iam.gserviceaccount.com" --role=roles/run.invoker --quiet
"""
    r = subprocess.run(["bash", "-ec", TB], env=os.environ, capture_output=True, text=True)
    print(r.stdout[-500:], r.stderr[-800:] if r.returncode else "")
    print("toolbox:", f"https://documind-toolbox-{NUMBER}.{REGION}.run.app/mcp" if r.returncode == 0 else "FAILED")


## Cell 9: The audit line


In [ ]:
# The audit line. Every call the server serves is one JSON log line - caller, tenant, tool, and for
# retrieve a hash of the query, never the query. 12.3's sink carries it to the warehouse.
r = subprocess.run(["gcloud", "logging", "read",
                    f'resource.type="cloud_run_revision" AND resource.labels.service_name="documind-mcp" AND jsonPayload.event="mcp_call"',
                    "--project", PROJECT_ID, "--limit", "5", "--format=value(jsonPayload.caller,jsonPayload.tenant,jsonPayload.tool,jsonPayload.answerable)"],
                   capture_output=True, text=True)
print(r.stdout or "(no rows yet - logs lag a minute; the three calls above will appear)")


## ✅ Lesson 7.2 complete

- ✅ `documind-mcp` deployed through the kit's own deploy block, as its own account, on the rosters
- ✅ Three calls: refused by the platform, refused by the roster, answered with citations
- ✅ Scale to zero measured, not asserted
- ✅ `make smoke-mcp` - the kit's fourth smoke leg
- ✅ The Toolbox as the second server, its query and its location fixed

**Next: 7.3 — an ADK agent over both servers, with a credential that refreshes itself.**
